# k-nearest neighbours

The simplest classifier there is: to label a new point, look at the labelled points nearest to it and let them vote. There is no training step and no weights. The whole model is the data plus a definition of "near". That makes it a good place to learn about distance, scaling, and what happens to geometry in many dimensions.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from common.datasets import (
    load_breast_cancer_data,
    load_digits_data,
    load_iris_flowers,
)
from common.plotting import plot_decision_boundary

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

You move to a new suburb and want to guess whether a house is worth a lot. You look at the five most similar houses nearby and take the majority. That is kNN with $k=5$.

Two choices define the model.

- **$k$, the number of neighbours.** With $k=1$ the model copies the single closest point, noise included, so the boundary is jagged and follows every quirk of the training set. With $k$ equal to the whole dataset every point gets the majority class. Small $k$ means low bias and high variance, large $k$ means the reverse. So $k$ plays the role the polynomial degree and the penalty played in notebook 13.
- **"Near".** A distance function. Whatever scale a feature is measured in decides how much it counts, which is why scaling matters so much here.

## 2. Maths

For a query $\mathbf x$, find the set $N_k(\mathbf x)$ of the $k$ training points with the smallest distance, then predict the most common label:

$$\hat y(\mathbf x)=\arg\max_c\sum_{i\in N_k(\mathbf x)}\mathbb 1[y_i=c]$$

Dividing the vote counts by $k$ gives a rough class probability. A refinement weights each vote by $1/d$, so closer neighbours count more.

**Distances.** The Minkowski family covers the usual ones, $d_p(\mathbf a,\mathbf b)=\bigl(\sum_j\lvert a_j-b_j\rvert^p\bigr)^{1/p}$.

| Metric | $p$ | When to use it |
|---|---|---|
| Euclidean | 2 | Default. Straight-line distance. |
| Manhattan | 1 | Less dominated by one large difference. Often better with many features. |
| Cosine | n/a | Compares direction, not length: $1-\frac{\mathbf a\cdot\mathbf b}{\lVert\mathbf a\rVert\lVert\mathbf b\rVert}$. Common for text. |

**Cost function.** There is none to minimise, because nothing is fitted. The thing we measure is the misclassification rate on held-out data, and we choose $k$ to reduce it. A classic result (Cover and Hart) says that as $n$ grows, 1-NN has an error at most twice the best possible error (the Bayes rate). That is a good guarantee for a method this simple, and it only holds when the data are dense enough, which is the problem in section 7.

**Cost of prediction.** Training is free, but each prediction compares the query with all $n$ points, $O(nd)$ in the brute-force case. Tree structures (KD-tree, ball tree) cut that to roughly $O(\log n)$ in low dimensions and lose the advantage in high ones.

## 3. From scratch

Brute force with numpy: compute all distances from the test points to the training points in one go, take the $k$ smallest per row, and vote.

In [ ]:
def knn_predict(
    X_train: np.ndarray, y_train: np.ndarray, X_query: np.ndarray, k: int = 5
) -> np.ndarray:
    # squared Euclidean distances, shape (n_query, n_train): |a-b|^2 = |a|^2 + |b|^2 - 2 a.b
    d2 = (
        (X_query**2).sum(axis=1)[:, None]
        + (X_train**2).sum(axis=1)[None, :]
        - 2 * X_query @ X_train.T
    )
    nearest = np.argpartition(d2, k - 1, axis=1)[:, :k]  # indices of the k smallest, unordered
    votes = y_train[nearest]
    return np.array([np.bincount(row).argmax() for row in votes])


X_iris, y_iris = load_iris_flowers()
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    X_iris.to_numpy(), y_iris.to_numpy(), test_size=0.3, random_state=SEED, stratify=y_iris
)
mine = knn_predict(Xi_tr, yi_tr, Xi_te, k=5)
ref = KNeighborsClassifier(n_neighbors=5).fit(Xi_tr, yi_tr).predict(Xi_te)
print("matches sklearn:", np.array_equal(mine, ref), " test accuracy:", accuracy_score(yi_te, mine))

## 4. Decision boundaries and the choice of k

Use two iris features so we can draw it. Compare $k=1$, $k=15$ and $k=60$ (60 is nearly all of the 105 training points).

In [ ]:
cols = ["petal_length", "sepal_width"]
X2 = X_iris[cols].to_numpy()
X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y_iris.to_numpy(), test_size=0.3, random_state=SEED, stratify=y_iris
)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, k in zip(axes, (1, 15, 60)):
    m = KNeighborsClassifier(n_neighbors=k).fit(X2_tr, y2_tr)
    plot_decision_boundary(m, X2_tr, y2_tr, ax, labels=("petal length", "sepal width"))
    ax.set_title(f"k={k}: train {m.score(X2_tr, y2_tr):.2f}, test {m.score(X2_te, y2_te):.2f}")
plt.show()

At $k=1$ the training accuracy is 1.0 by construction (each point is its own nearest neighbour) and the boundary is full of small islands. At $k=15$ the regions are smooth. At $k=60$ the model underfits: the small classes get outvoted.

Choose $k$ by cross-validation. A larger dataset makes the curve clearer, so use the breast cancer data (569 rows, 30 features), scaled inside a pipeline so no fold leaks.

In [ ]:
Xb, yb = load_breast_cancer_data()
ks = list(range(1, 41, 2))
scores = [
    cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)), Xb, yb, cv=10).mean()
    for k in ks
]
plt.plot(ks, scores, marker="o")
plt.xlabel("k")
plt.ylabel("10-fold CV accuracy")
plt.show()
print("best k:", ks[int(np.argmax(scores))], " accuracy:", round(max(scores), 4))

The curve is flat across a wide range. That is typical: pick a $k$ in the plateau rather than the single best value, and use an odd $k$ for two classes to avoid ties. Variance is high at the left edge, where $k=1$ is the least stable choice, and the score eases off as $k$ grows and bias rises.

Two more options. `weights="distance"` lets closer neighbours count more. `p=1` switches to Manhattan distance.

In [ ]:
for kw in [{}, {"weights": "distance"}, {"p": 1}]:
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(11, **kw))
    print(f"{kw!s:>22}: {cross_val_score(m, Xb, yb, cv=10).mean():.4f}")

## 5. Scaling

Distance adds up differences across features, so a feature measured in thousands drowns one measured in fractions. In the breast cancer data, `area` runs into the thousands while `smoothness` is about 0.1. Without scaling, nearly all the distance comes from `area` and a few other large-range columns.

In [ ]:
raw = KNeighborsClassifier(11)
scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(11))
print("unscaled CV accuracy:", cross_val_score(raw, Xb, yb, cv=10).mean().round(4))
print("scaled CV accuracy:  ", cross_val_score(scaled, Xb, yb, cv=10).mean().round(4))
print(
    Xb[["area error", "smoothness error", "mean area", "mean smoothness"]].std().round(3).to_dict()
)

Scaling lifts accuracy by a few points here. On data where the large-scale column is irrelevant, the unscaled model can fail completely. Always put a scaler in front of kNN, inside the pipeline.

## 6. Digits

A harder test: 1,797 handwritten digits, each an 8 by 8 grid of pixel intensities (64 features), ten classes. kNN does well because digits of the same class sit close together in pixel space.

In [ ]:
Xd, yd = load_digits_data()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=SEED, stratify=yd)
grid = GridSearchCV(
    KNeighborsClassifier(),
    {"n_neighbors": [1, 3, 5, 9, 15], "weights": ["uniform", "distance"]},
    cv=5,
).fit(Xd_tr, yd_tr)
print("best:", grid.best_params_, " CV accuracy:", round(grid.best_score_, 4))
print("test accuracy:", round(grid.score(Xd_te, yd_te), 4))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_estimator(grid, Xd_te, yd_te, ax=axes[0], colorbar=False)
wrong = np.flatnonzero(grid.predict(Xd_te) != yd_te)[:8]
axes[1].imshow(np.hstack([Xd_te.iloc[i].to_numpy().reshape(8, 8) for i in wrong]), cmap="gray_r")
axes[1].set_title(
    "first misclassified digits: " + " ".join(str(grid.predict(Xd_te.iloc[[i]])[0]) for i in wrong)
)
axes[1].axis("off")
plt.show()

About 98 to 99% accuracy with nothing but a vote. The errors are the ones a person would make: thin or oddly written digits. The scaler is not needed here because every pixel is on the same 0 to 16 scale, which is the exception to the rule above.

## 7. What breaks: the curse of dimensionality

kNN assumes that nearby points share a label. In high dimensions "nearby" stops meaning anything. Draw 500 random points in the unit cube and measure, for a random query, the nearest and farthest distances.

In [ ]:
rows = []
for d in (2, 10, 50, 200, 1000):
    pts = rng.uniform(size=(500, d))
    q = rng.uniform(size=d)
    dist = np.linalg.norm(pts - q, axis=1)
    rows.append(
        {
            "dimensions": d,
            "nearest": dist.min(),
            "farthest": dist.max(),
            "nearest / farthest": dist.min() / dist.max(),
        }
    )
pd.DataFrame(rows).set_index("dimensions").round(3)

In 2 dimensions the nearest point is a tiny fraction of the farthest. By 1,000 dimensions the ratio is close to 1: every point is about equally far away, so the "nearest" neighbours are barely nearer than anyone else. The volume of the space grows so fast that 500 points cannot fill it.

The practical version is irrelevant features. Add pure-noise columns to iris and watch kNN accuracy fall, while logistic regression, which can give the noise a weight near zero, copes better.

In [ ]:
from sklearn.linear_model import LogisticRegression

rows = []
for n_noise in (0, 10, 50, 200):
    noise = rng.normal(size=(len(X_iris), n_noise)) * X_iris.to_numpy().std()
    Xn = np.hstack([X_iris.to_numpy(), noise])
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(5))
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    rows.append(
        {
            "noise columns": n_noise,
            "kNN": cross_val_score(knn, Xn, y_iris, cv=5).mean(),
            "logistic": cross_val_score(lr, Xn, y_iris, cv=5).mean(),
        }
    )
pd.DataFrame(rows).set_index("noise columns").round(3)

kNN drops steadily as the noise columns outnumber the four real ones. The cure is fewer, better features: select them, or reduce dimensions first (see `04_unsupervised`).

**Other limits.** Prediction is slow on big data, because the whole training set must be kept and searched. Class imbalance hurts, since a rare class is outvoted in most neighbourhoods. kNN gives no weights to read, so it explains itself only through the neighbours it picked.

Next, a model that keeps only the few points that matter.